# add_messages 메시지 리듀서

사람과 모델의 메시지를 객체 형식으로 누적하면서 기존 메시지를 어떻게 수정할까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 실행합니다. 설치 셀은 이 노트북에 필요한 버전을 고정합니다.

## 패키지 설치와 버전 확인

필요한 패키지를 고정 버전으로 설치합니다. 출력에서 LangGraph 버전 또는 API 키의 존재 여부만 확인합니다.

In [1]:
# 노트북을 독립 실행할 수 있도록 패키지와 환경을 준비합니다.
%pip install -q langgraph==1.2.11 langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 scikit-learn==1.9.0
from importlib.metadata import version
print('Python 패키지 준비 완료')
print('langgraph', version('langgraph'))

Note: you may need to restart the kernel to use updated packages.
Python 패키지 준비 완료
langgraph 1.2.11


## 메시지 상태와 응답 노드 정의

다음 코드에서 메시지 상태와 응답 노드 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

In [2]:
# 메시지 상태와 응답 노드 정의에 사용할 상태 형식과 함수를 선언합니다.
from typing import Annotated
from typing_extensions import TypedDict
from langchain.messages import AnyMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def reply(state: ChatState) -> dict:
    question = state['messages'][-1].content
    return {'messages': [AIMessage(content=f'확인한 질문: {question}', id='a1')]}

## 메시지 그래프 실행

메시지 그래프 실행 단계를 실행합니다. 출력된 상태 값과 코드에서 반환한 키를 서로 대조합니다.

In [3]:
# 메시지 그래프 실행 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
builder = StateGraph(ChatState)
builder.add_node('reply', reply)
builder.add_edge(START, 'reply')
builder.add_edge('reply', END)
graph = builder.compile()
first = graph.invoke({'messages': [HumanMessage(content='대출 기간?', id='h1')]})
for message in first['messages']:
    print(message.type, message.id, message.content)

human h1 대출 기간?
ai a1 확인한 질문: 대출 기간?


## AIMessage.text 확인

AIMessage.text 확인 단계를 실행합니다. 출력된 상태 값과 코드에서 반환한 키를 서로 대조합니다.

In [4]:
# AIMessage.text 확인 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
answer_message = first['messages'][-1]
print('AIMessage.text:', answer_message.text)
assert answer_message.text == '확인한 질문: 대출 기간?'

AIMessage.text: 확인한 질문: 대출 기간?


## 같은 ID 메시지 교체 비교

같은 ID 메시지 교체 비교 조건으로 값을 하나만 변경합니다. 나란히 출력된 두 결과에서 달라진 항목을 확인합니다.

In [5]:
# 한 조건만 바꾸고 나머지 입력을 유지해 결과 차이를 비교합니다.
revised = add_messages(
    first['messages'],
    [HumanMessage(content='도서 대출 기간은?', id='h1')],
)
for message in revised:
    print(message.type, message.id, message.content)
assert len(revised) == 2
assert revised[0].content == '도서 대출 기간은?'

human h1 도서 대출 기간은?
ai a1 확인한 질문: 대출 기간?
